# Lab 3 · NumPy trên giá thật của Santiago

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành tuần 3**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo buổi 3 dùng các mảng nhỏ tự tạo để học cú pháp. Trong lab này, bạn áp
NumPy lên **17.688 mức giá thật** của Santiago — và lên ma trận review 4 năm × 12 tháng.

## Cách làm việc trong buổi lab

- Bài tập được chia bước; mỗi bước có ô `TODO` và phần kiểm tra `assert` — chạy qua hết
  `assert` nghĩa là bạn làm đúng.
- Phần khởi động và bài có hướng dẫn: bạn nên **tự gõ, không dùng AI** — các bài kiểm tra
  định kỳ 🚫 ở giờ lý thuyết đo đúng các kỹ năng này.
- Bài tự làm ở cuối được gắn nhãn ✅ mở: bạn được dùng AI, kèm trách nhiệm khai báo
  theo chính sách AI của môn.
- Bạn kẹt quá 3 phút ở một bước: gọi trợ giảng.

## Mục tiêu

Sau buổi lab, bạn:

1. Đưa được một cột dữ liệu thật vào NumPy và xử lý NaN chủ động.
2. Dùng thành thạo mặt nạ bool: đếm, tỷ lệ, lọc, ghép điều kiện.
3. Dùng `np.where`, percentile để gắn nhãn và tìm ngưỡng outlier.
4. Đọc đúng `axis` và `argmax` trên ma trận 2 chiều thật.

## Phần 0 · Khởi động (~10 phút)

Ba thao tác lõi của bài giảng, trên mảng nhỏ.

In [ ]:
import numpy as np

# W1 — tạo mảng và xem thuộc tính
a = np.array([4, 8, 15, 16, 23, 42])

# TODO: điền 2 thuộc tính của a
so_phan_tu = ...       # dùng a.shape hoặc a.size
kieu = ...             # dùng a.dtype (giữ nguyên object dtype, không đổi chuỗi)

# --- Ô kiểm tra ---
assert so_phan_tu == 6 and str(kieu) == "int64"
print("W1 ổn:", a.shape, a.dtype)

In [ ]:
# W2 — mặt nạ bool: đếm và tỷ lệ
diem = np.array([7.5, 4.0, 9.0, 5.5, 8.0, 3.0, 6.5, 9.5])

# TODO: đếm số điểm >= 5 và tỷ lệ điểm >= 8 (một biểu thức mỗi dòng)
so_dat = ...
ty_le_gioi = ...

# --- Ô kiểm tra ---
assert so_dat == 6 and ty_le_gioi == 0.375
print(f"W2 ổn: {so_dat} bài đạt, {ty_le_gioi:.0%} giỏi")

In [ ]:
# W3 — axis trên ma trận 2x3
M = np.array([[1, 2, 3],
              [4, 5, 6]])

# TODO: tổng theo CỘT (một số cho mỗi cột) và tổng theo HÀNG
tong_cot = ...
tong_hang = ...

# --- Ô kiểm tra ---
assert list(tong_cot) == [5, 7, 9] and list(tong_hang) == [6, 15]
print("W3 ổn — axis nào bị gọi tên, chiều đó biến mất.")

## Phần 1 · Giá thật vào NumPy (~40 phút)

Ta lấy cột giá bằng một dòng pandas (buổi 4 học kỹ) rồi làm việc thuần NumPy từ đó.

In [ ]:
import pandas as pd

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/visualisations/listings.csv")
gia = pd.read_csv(URL)["price"].to_numpy()

gia.dtype, gia.shape

### Bước 1 · NaN — dọn chủ động trước khi tính

Cột giá là `float64` và chứa `nan` (các phòng không khai giá — bạn đã gặp chúng ở lab 2).

In [ ]:
# TODO: đếm số NaN bằng np.isnan, rồi tạo mảng gia_sach không còn NaN
so_nan = ...
gia_sach = ...          # gợi ý: mặt nạ ~np.isnan(gia)

# --- Ô kiểm tra ---
assert so_nan == 846 and gia_sach.shape == (17688,)
assert not np.isnan(gia_sach).any()
print(f"Bỏ {so_nan} NaN, còn {gia_sach.size:,} giá hợp lệ.")

Vì sao không dùng luôn `np.nanmean` cho mọi thứ? Vì các phép sau (lọc, phần trăm,
nhãn) đều cần mảng sạch — dọn một lần ở đầu, mọi bước sau đơn giản hẳn. Đó cũng chính là
lý do pipeline có bước làm sạch riêng.

### Bước 2 · Câu hỏi thị trường bằng mặt nạ bool

In [ ]:
# TODO: trả lời 3 câu bằng mặt nạ bool trên gia_sach
ty_le_tren_100k = ...     # tỷ lệ phòng giá > 100.000 CLP (mean trên bool)
so_duoi_30k = ...         # số phòng giá < 30.000 CLP
so_khoang_50_100 = ...    # số phòng 50.000 <= giá <= 100.000 (ghép 2 điều kiện bằng &)

# --- Ô kiểm tra ---
assert round(ty_le_tren_100k, 4) == 0.2088
assert so_duoi_30k == 1660
assert so_khoang_50_100 == (( (gia_sach >= 50_000) & (gia_sach <= 100_000) ).sum())
print(f"{ty_le_tren_100k:.1%} trên 100k · {so_duoi_30k} phòng dưới 30k")

### Bước 3 · np.where — gắn nhãn cả mảng một lệnh

In [ ]:
med = np.median(gia_sach)          # 59000.0

# TODO: tạo mảng nhan: "cao" nếu giá > med, ngược lại "pho thong"
nhan = ...

# --- Ô kiểm tra ---
assert med == 59000.0
assert (nhan == "cao").sum() == 8841
print(f"Trung vị {med:,.0f} CLP — {(nhan == 'cao').sum():,} phòng thuộc nửa 'cao'.")

Để ý: "nửa cao" chỉ có 8.841 phòng — chưa đúng một nửa của 17.688. Vì sao?
Có 4 phòng giá **đúng bằng** trung vị, chúng rơi vào nhánh "pho thong". Chi tiết nhỏ,
nhưng đúng kiểu câu hỏi vấn đáp: ranh giới thuộc về bên nào là **quyết định của bạn**.

### Bước 4 · Percentile — ngưỡng outlier đầu tiên của bạn

In [ ]:
# TODO: tính P1, P99 của gia_sach (np.percentile) và đếm số giá nằm NGOÀI [P1, P99]
p1, p99 = ...
so_ngoai = ...            # gợi ý: (gia_sach < p1) | (gia_sach > p99)

# --- Ô kiểm tra ---
assert round(p1) == 16369 and round(p99) == 686144
assert so_ngoai == 354
print(f"Khoảng [P1, P99] = [{p1:,.0f}, {p99:,.0f}] — {so_ngoai} giá nằm ngoài (~2%).")

P99 ≈ 686 nghìn CLP — mọi giá trên ngưỡng này (kể cả phòng 97 triệu của lab 2) là ứng
viên gắn cờ ở buổi 10. Ngưỡng lấy **từ chính dữ liệu**, không phải con số bịa.

## Phần 2 · Ma trận review 4 năm × 12 tháng (~20 phút)

Bảng dưới là số review Santiago đếm theo năm (2022–2025) và tháng (1–12) — số thật,
trích từ 690 nghìn review của lab 2.

In [ ]:
M = np.array([
    [ 3231,  2854,  3469,  3449,  3536,  3155,  4580,  4509,  4963,  4951,  5100,  4590],
    [ 5315,  4584,  5555,  5186,  5066,  5122,  6997,  7440,  6503,  7528,  8523,  7371],
    [ 8218,  6959,  9330,  9997,  8299,  9705, 13066, 13659, 11722, 12480, 15176, 12656],
    [14636, 11652, 16565, 14041, 14509, 15192, 21726, 22904, 17301, 20791, 23028, 19087],
])  # hàng = 2022, 2023, 2024, 2025; cột = tháng 1..12
nam = np.array([2022, 2023, 2024, 2025])

M.shape

In [ ]:
# TODO: dùng đúng axis để tính
tong_moi_nam = ...        # tổng review từng năm (4 số)
tb_moi_thang = ...        # trung bình mỗi tháng, gộp 4 năm (12 số)

# --- Ô kiểm tra ---
assert list(tong_moi_nam) == [48387, 75190, 131267, 211432]
assert round(float(tb_moi_thang[0])) == 7850
print("Năm 2025 gấp ~4.4 lần năm 2022 — thị trường phục hồi rất nhanh sau COVID.")

In [ ]:
# TODO: argmax hai chiều
thang_dinh = ...          # tháng cao nhất của TỪNG năm (4 số, tính theo 1..12 — nhớ +1)
nam_dinh_thang1 = ...     # NĂM có tháng-1 cao nhất (dùng nam[...])

# --- Ô kiểm tra ---
assert list(thang_dinh) == [11, 11, 11, 11]
assert nam_dinh_thang1 == 2025
print("Cả 4 năm đều đỉnh vào tháng 11 (theo đếm thô).")

Câu hỏi đáng ngẫm: bảng đếm thô này trộn lẫn **hai tín hiệu** — mùa vụ trong năm và
đà tăng trưởng của cả thị trường. Muốn nhìn riêng "mùa vụ", hãy chuẩn hoá trong từng năm
(chia mỗi hàng cho trung bình của chính hàng đó) rồi xem tháng nào thật sự cao. Chọn cách
đo cũng là một quyết định phân tích — buổi 8 sẽ quay lại chủ đề này.

## Phần 3 · Bài tự làm ✅ mở (làm xong sớm / về nhà)

Được dùng AI theo quy trình 5 bước; ghi lại prompt chính + cách bạn kiểm chứng.

### Tự làm 1 · Top 10 giá bằng argsort

Dùng `np.argsort` trên `gia_sach` để lấy **10 giá cao nhất** (không dùng pandas).
In ra và đối chiếu: giá cao nhất có đúng 97.000.045 như lab 2 không?

### Tự làm 2 · Bootstrap trung vị (nâng cao)

Trung vị 59.000 CLP "chắc" đến đâu? Dùng `rng = np.random.default_rng(42)`:
lặp 1.000 lần việc *lấy mẫu lại có hoàn lại* (`rng.choice(gia_sach, size=gia_sach.size,
replace=True)`) và tính trung vị mỗi lần; lấy `np.percentile(..., [2.5, 97.5])` của
1.000 trung vị đó. Khoảng thu được nói lên điều gì? (Kỹ thuật này tên là bootstrap —
sẽ hữu ích khi bạn muốn nói "con số này ± bao nhiêu" trong báo cáo bài tập lớn.)

In [ ]:
# Viết bài tự làm của bạn ở đây

## Tóm tắt buổi lab

| Bạn đã làm | Sẽ gặp lại ở |
|---|---|
| NaN: đếm bằng isnan, dọn chủ động | missing values (buổi 10) |
| Mặt nạ bool: đếm / tỷ lệ / ghép & | lọc pandas (buổi 4–5), KPI bài tập lớn |
| np.where gắn nhãn; ranh giới là quyết định | gắn cờ QA (buổi 10) |
| Percentile làm ngưỡng outlier từ dữ liệu | bộ quy tắc QA (buổi 10) |
| axis & argmax trên ma trận thật | pivot_table (buổi 5), resample (buổi 8) |

Buổi lý thuyết tới: **pandas** — mảng có tên cột, và mọi kỹ năng NumPy dùng lại nguyên vẹn.